# Guitar Pedalboard One Cell

A single-cell pedalboard UI for the live PYNQ-Z2 effect chain.
Built on top of `GuitarEffectSwitcher.ipynb` but consolidates every
effect into one ipywidgets panel, and drives the new pedal-mask
distortion API instead of the legacy `distortion=`/`distortion_*`
knobs.

## Effect chain

```
Noise Suppressor -> Overdrive -> Distortion Pedalboard
             -> Amp Simulator -> Cab IR -> EQ -> Reverb
```

## Distortion pedals

- **Implemented in the current bitstream**: `clean_boost`,
  `tube_screamer`, `rat` (mapped onto the existing RAT stage),
  `metal`.
- **Reserved (bypass stub)**: `ds1`, `big_muff`, `fuzz_face`.
  Selecting them sets the corresponding pedal-mask bit and prints a
  warning, but the FPGA stage currently leaves audio bit-exact.

## Loud-output warning

- Drop headphone volume **before** clicking Apply.
- Start at `level = 30..35`. Crank only after you have heard the
  preset at low volume.
- `rat` and `metal` are very high-gain. Stacking pedals
  (`exclusive=False`) is louder still — the cell auto-trims `level`
  to 25 when you turn stack mode on.

## ADC HPF default-on

`config_codec()` enables the ADAU1761 ADC digital HPF, so after
load `R19_ADC_CONTROL == 0x23`. That filter is a ~2 Hz DC blocker,
**not** a 20-40 Hz guitar low-cut. The status panel below confirms
the value on every Apply / Refresh.

## What this notebook does *not* change

Notebook-only update. No bitstream rebuild needed; the active
bit/hwh on the board is the pedal-mask build at
`baa97ff` and remains untouched.


In [ ]:
import sys
import traceback
from pathlib import Path

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import ipywidgets as widgets
from IPython.display import display, clear_output

from audio_lab_pynq.AudioLabOverlay import AudioLabOverlay

# Import shared presets / defaults from the package, falling back to
# inline copies if an older deployed package does not yet have those
# modules. The fallback values are byte-for-byte identical to the
# package source; if you change one here, change the other in the
# same commit. See docs/ai_context/EFFECT_ADDING_GUIDE.md.
try:
    from audio_lab_pynq.effect_presets import (
        DISTORTION_PRESETS as PRESETS,
        NOISE_SUPPRESSOR_PRESETS as NS_PRESETS,
    )
except Exception:
    PRESETS = {
        "Clean Boost": dict(distortion_on=True, pedal="clean_boost",
                            drive=35, tone=50, level=45,
                            bias=50, tight=50, mix=100),
        "Tube Screamer Crunch": dict(distortion_on=True, pedal="tube_screamer",
                                     drive=45, tone=55, level=35,
                                     bias=50, tight=60, mix=100),
        "RAT Distortion": dict(distortion_on=True, pedal="rat",
                               drive=55, tone=45, level=35,
                               bias=50, tight=50, mix=100),
        "Metal Tight": dict(distortion_on=True, pedal="metal",
                            drive=55, tone=55, level=30,
                            bias=50, tight=75, mix=100),
    }
    NS_PRESETS = {
        "NS-2 Style":        dict(threshold=35, decay=45, damp=80),
        "NS-1X Natural":     dict(threshold=30, decay=55, damp=60),
        "High Gain Tight":   dict(threshold=55, decay=20, damp=90),
        "Sustain Friendly":  dict(threshold=25, decay=75, damp=45),
    }

# ---- output console (used everywhere below) -------------------------
status = widgets.Output(layout=widgets.Layout(border="1px solid #ccc",
                                              max_height="320px",
                                              overflow="auto"))

# ---- reserved-pedal helper ------------------------------------------
PEDAL_LABEL_TO_API = {
    "off":                 None,
    "clean_boost":         "clean_boost",
    "tube_screamer":       "tube_screamer",
    "rat":                 "rat",
    "metal":               "metal",
    "ds1_reserved":        "ds1",
    "big_muff_reserved":   "big_muff",
    "fuzz_face_reserved":  "fuzz_face",
}
RESERVED_PEDALS = {"ds1", "big_muff", "fuzz_face"}

# ---- widget factory helpers -----------------------------------------
# Reused by every effect section. Add a new effect section by calling
# make_slider / make_section instead of repeating the layout boilerplate.
SLIDER_LAYOUT = widgets.Layout(width="380px")
SLIDER_STYLE = {"description_width": "120px"}


def make_slider(description, value, minimum=0, maximum=100):
    """An IntSlider with the standard one-cell-UI layout / style."""
    return widgets.IntSlider(value=value, min=minimum, max=maximum,
                             description=description,
                             style=SLIDER_STYLE, layout=SLIDER_LAYOUT)


def make_section(*children):
    """Wrap an effect section\'s widgets into a VBox suitable for the
    main accordion."""
    return widgets.VBox(list(children))


# ---- overlay initialisation -----------------------------------------
ovl = None
try:
    ovl = AudioLabOverlay()
    # Always start from a known-quiet state.
    ovl.clear_distortion_pedals()
    ovl.set_distortion_settings(drive=20, tone=50, level=35,
                                bias=50, tight=50, mix=100)
    ovl.set_guitar_effects(distortion_on=False)
except Exception:
    with status:
        print("AudioLabOverlay() failed:")
        traceback.print_exc()


def _say(*args):
    with status:
        print(*args)


# ---- helpers --------------------------------------------------------
def refresh_status(*_):
    if ovl is None:
        return
    with status:
        clear_output()
        try:
            print("ADC HPF        :", ovl.codec.get_adc_hpf_state())
            print("R19_ADC_CONTROL:", hex(ovl.codec.R19_ADC_CONTROL[0]))
            s = ovl.get_distortion_settings()
            print("pedal_mask     : 0x{:02x}".format(s["pedal_mask"]))
            print("active pedals  :",
                  [k for k, v in s["pedals"].items() if v] or "(none)")
            print("dist params    : drive={drive} tone={tone} level={level}"
                  " bias={bias} tight={tight} mix={mix}".format(**s))
            try:
                ns_state = ovl.get_noise_suppressor_settings()
                print("noise suppressor: enabled={enabled} threshold={threshold}"
                      " decay={decay} damp={damp}".format(**ns_state))
                print("ns bytes       : threshold_byte={threshold_byte}"
                      " decay_byte={decay_byte} damp_byte={damp_byte}"
                      " mode={mode}".format(**ns_state))
                print("ns gpio        : {gpio_name}"
                      " has_gpio={has_gpio}"
                      " reflected_to_fpga={reflected_to_fpga}"
                      " status={implementation_status}".format(**ns_state))
            except AttributeError:
                print("noise suppressor: API not available on this overlay")
        except Exception:
            traceback.print_exc()


def _selected_pedals(stack_mode, dropdown_value, multiselect_value):
    """Translate the dropdown / multi-select widgets into the list of
    API pedal names to enable. Returns (pedal_names, warning_lines)."""
    warnings = []
    if stack_mode:
        names = []
        for label in multiselect_value:
            api = PEDAL_LABEL_TO_API[label]
            if api is None:
                continue
            names.append(api)
            if api in RESERVED_PEDALS:
                warnings.append("note: {} is reserved; FPGA stage is a "
                                "bypass stub in the current bitstream"
                                .format(api))
        return names, warnings
    api = PEDAL_LABEL_TO_API[dropdown_value]
    if api is None:
        return [], warnings
    if api in RESERVED_PEDALS:
        warnings.append("note: {} is reserved; FPGA stage is a bypass "
                        "stub in the current bitstream".format(api))
    return [api], warnings


def apply_distortion_settings(master_on, pedal_names, exclusive, stack_mode):
    """Push the distortion-section widget state to the overlay."""
    if stack_mode and w_dist_level.value > 25:
        _say("stack mode: auto-trimming distortion level "
             "{} -> 25 to avoid loud surprises".format(w_dist_level.value))
        w_dist_level.value = 25
    if not master_on or not pedal_names:
        ovl.clear_distortion_pedals()
        ovl.set_distortion_settings(
            drive=w_dist_drive.value, tone=w_dist_tone.value,
            level=w_dist_level.value, bias=w_dist_bias.value,
            tight=w_dist_tight.value, mix=w_dist_mix.value,
        )
        return
    if exclusive:
        ovl.set_distortion_settings(
            pedal=pedal_names[0], exclusive=True,
            drive=w_dist_drive.value, tone=w_dist_tone.value,
            level=w_dist_level.value, bias=w_dist_bias.value,
            tight=w_dist_tight.value, mix=w_dist_mix.value,
        )
    else:
        ovl.clear_distortion_pedals()
        ovl.set_distortion_pedals(**{p: True for p in pedal_names})
        ovl.set_distortion_settings(
            drive=w_dist_drive.value, tone=w_dist_tone.value,
            level=w_dist_level.value, bias=w_dist_bias.value,
            tight=w_dist_tight.value, mix=w_dist_mix.value,
        )


def apply_noise_suppressor_settings():
    """Push the Noise Suppressor widget state to the dedicated GPIO."""
    return ovl.set_noise_suppressor_settings(
        enabled=w_gate_on.value,
        threshold=w_ns_threshold.value,
        decay=w_ns_decay.value,
        damp=w_ns_damp.value,
    )


def apply_chain_settings(master_on):
    """Push every other section (overdrive, amp, cab, eq, reverb) plus
    the master noise-gate / distortion-on flags via set_guitar_effects."""
    return ovl.set_guitar_effects(
        noise_gate_on=w_gate_on.value,
        noise_gate_threshold=w_ns_threshold.value,
        overdrive_on=w_od_on.value,
        overdrive_drive=w_od_drive.value,
        overdrive_tone=w_od_tone.value,
        overdrive_level=w_od_level.value,
        distortion_on=master_on,
        amp_on=w_amp_on.value,
        amp_input_gain=w_amp_gain.value,
        amp_bass=w_amp_bass.value,
        amp_middle=w_amp_mid.value,
        amp_treble=w_amp_treble.value,
        amp_presence=w_amp_presence.value,
        amp_resonance=w_amp_resonance.value,
        amp_master=w_amp_master.value,
        amp_character=w_amp_character.value,
        cab_on=w_cab_on.value,
        cab_mix=w_cab_mix.value,
        cab_level=w_cab_level.value,
        cab_model=w_cab_model.value,
        cab_air=w_cab_air.value,
        eq_on=w_eq_on.value,
        eq_low=w_eq_low.value,
        eq_mid=w_eq_mid.value,
        eq_high=w_eq_high.value,
        reverb_on=w_rev_on.value,
        reverb_decay=w_rev_decay.value,
        reverb_tone=w_rev_tone.value,
        reverb_mix=w_rev_mix.value,
    )


def apply_settings(*_):
    if ovl is None:
        _say("overlay not initialised; nothing to apply")
        return
    try:
        with status:
            clear_output()

        master_on = w_dist_master.value
        stack_mode = w_dist_stack.value
        exclusive = w_dist_exclusive.value and not stack_mode
        pedal_names, warns = _selected_pedals(stack_mode,
                                              w_dist_pedal.value,
                                              w_dist_multi.value)
        for line in warns:
            _say(line)

        apply_distortion_settings(master_on, pedal_names, exclusive, stack_mode)
        apply_noise_suppressor_settings()
        apply_chain_settings(master_on)

        active = [n for n, on in [
            ("Noise Gate", w_gate_on.value),
            ("Overdrive", w_od_on.value),
            ("Distortion Pedalboard", master_on and bool(pedal_names)),
            ("Amp", w_amp_on.value),
            ("Cab IR", w_cab_on.value),
            ("EQ", w_eq_on.value),
            ("Reverb", w_rev_on.value),
        ] if on]
        _say("applied; active stages:", " -> ".join(active) or "(passthrough)")
        if pedal_names:
            _say("distortion pedals:", pedal_names)
        refresh_status()
    except Exception:
        with status:
            traceback.print_exc()


def safe_bypass(*_):
    if ovl is None:
        _say("overlay not initialised; nothing to bypass")
        return
    try:
        with status:
            clear_output()
        ovl.clear_distortion_pedals()
        ovl.set_distortion_settings(drive=20, tone=50, level=35,
                                    bias=50, tight=50, mix=100)
        ovl.set_noise_suppressor_settings(enabled=False)
        ovl.set_guitar_effects(
            noise_gate_on=False,
            overdrive_on=False,
            distortion_on=False,
            rat_on=False,
            amp_on=False,
            cab_on=False,
            eq_on=False,
            reverb_on=False,
        )
        # Reset the UI checkboxes / sliders to the safe defaults too.
        w_gate_on.value = False
        w_ns_threshold.value = 35
        w_ns_decay.value = 40
        w_ns_damp.value = 70
        w_od_on.value = False
        w_dist_master.value = False
        w_dist_pedal.value = "off"
        w_dist_multi.value = ()
        w_dist_stack.value = False
        w_dist_exclusive.value = True
        w_dist_drive.value = 20
        w_dist_tone.value = 50
        w_dist_level.value = 35
        w_dist_bias.value = 50
        w_dist_tight.value = 50
        w_dist_mix.value = 100
        w_amp_on.value = False
        w_cab_on.value = False
        w_eq_on.value = False
        w_rev_on.value = False
        _say("safe bypass complete")
        refresh_status()
    except Exception:
        with status:
            traceback.print_exc()


def apply_ns_preset(name):
    spec = NS_PRESETS[name]
    w_gate_on.value = True
    w_ns_threshold.value = spec["threshold"]
    w_ns_decay.value = spec["decay"]
    w_ns_damp.value = spec["damp"]
    apply_settings()


def apply_preset(name):
    spec = PRESETS[name]
    w_dist_master.value = bool(spec.get("distortion_on", True))
    w_dist_stack.value = False
    w_dist_exclusive.value = True
    w_dist_pedal.value = spec["pedal"]
    w_dist_drive.value = spec["drive"]
    w_dist_tone.value = spec["tone"]
    w_dist_level.value = spec["level"]
    w_dist_bias.value = spec["bias"]
    w_dist_tight.value = spec["tight"]
    w_dist_mix.value = spec["mix"]
    apply_settings()


# ---- widgets --------------------------------------------------------
# Noise Suppressor (BOSS NS-2 / NS-1X style; FPGA THRESHOLD/DECAY/DAMP).
# The enable flag still rides on the legacy noise_gate_on bit so older
# bitstreams keep working; the active block in the new bitstream is the
# noise_suppressor_control GPIO at 0x43CC0000.
w_gate_on = widgets.Checkbox(value=False, description="Noise Suppressor",
                             indent=False)
w_ns_threshold = make_slider("Threshold", 35)
w_ns_decay = make_slider("Decay", 40)
w_ns_damp = make_slider("Damp", 70)
# Legacy alias kept so older code in this cell that referenced
# w_gate_thr still works at import time. The actual byte sent to the
# bitstream comes from the new threshold / decay / damp sliders above.
w_gate_thr = w_ns_threshold

# Overdrive
w_od_on = widgets.Checkbox(value=False, description="Overdrive", indent=False)
w_od_drive = make_slider("Drive", 30)
w_od_tone = make_slider("Tone", 65)
w_od_level = make_slider("Level", 100, 0, 200)

# Distortion pedalboard
w_dist_master = widgets.Checkbox(value=False, description="Distortion master",
                                 indent=False)
w_dist_pedal = widgets.Dropdown(
    options=["off", "clean_boost", "tube_screamer", "rat", "metal",
             "ds1_reserved", "big_muff_reserved", "fuzz_face_reserved"],
    value="off", description="Pedal",
    style=SLIDER_STYLE, layout=widgets.Layout(width="380px"))
w_dist_exclusive = widgets.Checkbox(value=True, description="Exclusive",
                                    indent=False)
w_dist_stack = widgets.Checkbox(value=False, description="Stack mode (advanced)",
                                indent=False)
w_dist_multi = widgets.SelectMultiple(
    options=["clean_boost", "tube_screamer", "rat", "metal",
             "ds1_reserved", "big_muff_reserved", "fuzz_face_reserved"],
    value=(), description="Stacked",
    style=SLIDER_STYLE, layout=widgets.Layout(width="380px", height="120px"))
w_dist_drive = make_slider("Drive", 20)
w_dist_tone = make_slider("Tone", 50)
w_dist_level = make_slider("Level", 35)
w_dist_bias = make_slider("Bias", 50)
w_dist_tight = make_slider("Tight", 50)
w_dist_mix = make_slider("Mix", 100)

# Amp simulator
w_amp_on = widgets.Checkbox(value=False, description="Amp Simulator",
                            indent=False)
w_amp_gain = make_slider("Gain", 35)
w_amp_bass = make_slider("Bass", 50)
w_amp_mid = make_slider("Middle", 50)
w_amp_treble = make_slider("Treble", 50)
w_amp_presence = make_slider("Presence", 45)
w_amp_resonance = make_slider("Resonance", 35)
w_amp_master = make_slider("Master", 80, 0, 150)
w_amp_character = make_slider("Character", 35)

# Cab IR
w_cab_on = widgets.Checkbox(value=False, description="Cab IR", indent=False)
w_cab_mix = make_slider("Mix", 100)
w_cab_level = make_slider("Level", 100, 0, 150)
w_cab_model = widgets.Dropdown(
    options=[("1x12 open back", 0), ("2x12 british", 1), ("4x12 closed", 2)],
    value=1, description="Model",
    style=SLIDER_STYLE, layout=widgets.Layout(width="380px"))
w_cab_air = make_slider("Air", 50)

# EQ
w_eq_on = widgets.Checkbox(value=False, description="EQ", indent=False)
w_eq_low = make_slider("Low", 100, 0, 200)
w_eq_mid = make_slider("Mid", 100, 0, 200)
w_eq_high = make_slider("High", 100, 0, 200)

# Reverb
w_rev_on = widgets.Checkbox(value=False, description="Reverb", indent=False)
w_rev_decay = make_slider("Decay", 30)
w_rev_tone = make_slider("Tone", 65)
w_rev_mix = make_slider("Mix", 20)

# Top-row buttons
b_apply = widgets.Button(description="Apply", button_style="primary")
b_bypass = widgets.Button(description="Safe Bypass", button_style="warning")
b_refresh = widgets.Button(description="Refresh Status")
b_apply.on_click(apply_settings)
b_bypass.on_click(safe_bypass)
b_refresh.on_click(refresh_status)

preset_buttons = []
for name in PRESETS:
    btn = widgets.Button(description=name)
    btn.on_click(lambda _b, n=name: apply_preset(n))
    preset_buttons.append(btn)

# Toggle Distortion stack-mode UI: hide single dropdown when stacking.
def _on_stack_change(change):
    stacking = bool(change["new"])
    w_dist_pedal.layout.display = "none" if stacking else None
    w_dist_exclusive.layout.display = "none" if stacking else None
    w_dist_multi.layout.display = None if stacking else "none"
w_dist_stack.observe(_on_stack_change, names="value")
_on_stack_change({"new": w_dist_stack.value})

# ---- layout ---------------------------------------------------------
title = widgets.HTML(
    "<h2 style=\'margin-bottom:4px\'>Guitar Pedalboard One Cell</h2>"
    "<div style=\'color:#666;margin-bottom:8px\'>"
    "Noise Gate -> Overdrive -> Distortion Pedalboard -> "
    "Amp -> Cab -> EQ -> Reverb"
    "</div>")

button_row = widgets.HBox([b_apply, b_bypass, b_refresh])
preset_row = widgets.HBox(preset_buttons)

dist_box = make_section(
    widgets.HBox([w_dist_master, w_dist_stack, w_dist_exclusive]),
    w_dist_pedal,
    w_dist_multi,
    w_dist_drive, w_dist_tone, w_dist_level,
    w_dist_bias, w_dist_tight, w_dist_mix,
)

ns_section = make_section(
    widgets.HTML("<div style=\'color:#666;font-size:90%\'>"
                 "BOSS NS-2 / NS-1X style noise suppressor (FPGA"
                 " THRESHOLD / DECAY / DAMP). New 0-100 threshold"
                 " scale: 100 \u2261 legacy 10 (byte = round(threshold"
                 " * 255 / 1000)).</div>"),
    w_gate_on, w_ns_threshold, w_ns_decay, w_ns_damp,
    widgets.HBox([widgets.Button(description=n,
                                 layout=widgets.Layout(width="170px"))
                  for n in NS_PRESETS]),
)

acc = widgets.Accordion(children=[
    ns_section,
    make_section(w_od_on, w_od_drive, w_od_tone, w_od_level),
    dist_box,
    make_section(w_amp_on, w_amp_gain, w_amp_bass, w_amp_mid,
                 w_amp_treble, w_amp_presence, w_amp_resonance,
                 w_amp_master, w_amp_character),
    make_section(w_cab_on, w_cab_mix, w_cab_level, w_cab_model, w_cab_air),
    make_section(w_eq_on, w_eq_low, w_eq_mid, w_eq_high),
    make_section(w_rev_on, w_rev_decay, w_rev_tone, w_rev_mix),
])
for i, name in enumerate(["Noise Suppressor", "Overdrive",
                          "Distortion Pedalboard", "Amp Simulator",
                          "Cab IR", "EQ", "Reverb"]):
    acc.set_title(i, name)
acc.selected_index = 2  # open Distortion Pedalboard by default

# Wire the NS preset buttons we built inside the accordion. The HBox at
# index -1 of the Noise Suppressor VBox holds them.
ns_button_row = acc.children[0].children[-1]
for btn, preset_name in zip(ns_button_row.children, NS_PRESETS):
    btn.on_click(lambda _b, n=preset_name: apply_ns_preset(n))

display(widgets.VBox([title, button_row, preset_row, status, acc]))
refresh_status()
